In [ ]:
import pandas as pd
import pyodbc
 
# --------------------------------------------------------------------
# 1. Paramètres
# --------------------------------------------------------------------
FICHIER = r"C:\Users\stgadmin\Desktop\TFE-STIB\stib_line_exploitation_1.xlsx" 
SERVEUR = "ICT-202-11"                           
BASE    = "TFE_STIB"
TABLE   = "dbo.staging_line_exploitation"
 
CHAINE = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVEUR};"
    f"DATABASE={BASE};"
    "Trusted_Connection=yes;"
)
 
# --------------------------------------------------------------------
# 2. Lecture du fichier — tout en texte
# --------------------------------------------------------------------
df = pd.read_excel(FICHIER, sheet_name="exploitation", dtype=str)
 
# Nettoyage minimal : espaces parasites, chaînes vides -> NaN
for col in ["line_id", "type_exploitation"]:
    df[col] = df[col].str.strip().replace("", pd.NA)
 
# --------------------------------------------------------------------
# 3. Contrôles AVANT insertion (on ne charge pas des données douteuses)
# --------------------------------------------------------------------
assert df["line_id"].notna().all(),           "line_id manquant"
assert df["type_exploitation"].notna().all(), "type_exploitation manquant"
assert not df["line_id"].duplicated().any(),  "line_id en double"
 
print(f"{len(df)} lignes lues")
print(df["type_exploitation"].value_counts().to_string())
 
donnees = list(df[["line_id", "type_exploitation"]].itertuples(index=False, name=None))
 
# --------------------------------------------------------------------
# 4. Chargement
# --------------------------------------------------------------------
with pyodbc.connect(CHAINE, autocommit=False) as cnx:
    cur = cnx.cursor()
    cur.fast_executemany = True
 
    cur.execute(f"TRUNCATE TABLE {TABLE};")          # rejouable
    cur.executemany(
        f"INSERT INTO {TABLE} (line_id, type_exploitation) VALUES (?, ?);",
        donnees,
    )
    cnx.commit()
 
    cur.execute(f"SELECT COUNT(*) FROM {TABLE};")
    print("Lignes en base :", cur.fetchone()[0])